# Fire CNN training (MobileNetV2) on D-Fire - Colab GPU

Trains `models/fire_cnn.keras` for the Predictive AI Hazard Prevention System.
Target on the **test** split: fire-vs-rest F1 >= 0.90 and recall >= 0.92.
Report the real numbers even if the target is not met. Never tune on the test split.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. This notebook has not been run by its author:
run it once and fix any path that differs on your account.

## 1. Get the project code onto Colab
Upload `hazard_system.zip` (the project folder zipped) to the Colab Files panel, or to Google Drive and mount it.

In [ ]:
from google.colab import files
up = files.upload()          # choose hazard_system.zip

In [ ]:
!unzip -q -o hazard_system.zip -d /content
%cd /content/hazard_system
!pip install -q tensorflow==2.18.0 opencv-python pillow scikit-learn pandas kagglehub
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Get D-Fire
D-Fire (21,000+ images, YOLO labels; class ids 0 = smoke, 1 = fire) is published by GAIA: https://github.com/gaiasd/DFireDataset
The authors give OneDrive links and a ready-to-use Kaggle copy (`sayedgamal99/smoke-fire-detection-yolo`).

**Option A (Kaggle).** Needs a free Kaggle account. Run the cell and sign in when asked.
**Option B (manual).** Download from the OneDrive link in the GAIA README, upload to Drive, unzip, and set `DFIRE_ROOT` to the folder.
Check the dataset licence and keep a note of the version you used.

In [ ]:
import kagglehub
DFIRE_ROOT = kagglehub.dataset_download("sayedgamal99/smoke-fire-detection-yolo")
print(DFIRE_ROOT)
# Option B example:  DFIRE_ROOT = "/content/drive/MyDrive/DFire"

In [ ]:
import os
n_img = sum(len(f) for _, _, f in os.walk(DFIRE_ROOT))
print("files under root:", n_img)
for r, d, f in os.walk(DFIRE_ROOT):
    print(r, len(f)); 
    if r.count(os.sep) - DFIRE_ROOT.count(os.sep) >= 2: d[:] = []

## 3. Build the leak-safe manifest
The script re-splits images by near-duplicate group (not by D-Fire's own folders) so near-identical frames cannot appear in both train and test.
It expects `<any folder>/images/*.jpg` with sibling `<any folder>/labels/*.txt`. It needs `--fire-id 1 --smoke-id 0` for D-Fire.

In [ ]:
!python -m ml.prepare_fire_dataset --root "$DFIRE_ROOT" --out data/fire_manifest.csv --fire-id 1 --smoke-id 0 --seed 42
import pandas as pd
m = pd.read_csv("data/fire_manifest.csv"); print(m.shape); print(m.head()); print(m.groupby(["split"]).size())

## 4. Train (two stages: head, then fine-tune the top 30 layers)
Defaults: 224x224 images, batch 32, 5 head epochs, 5 fine-tune epochs, ImageNet weights, seed 42.

In [ ]:
!python -m ml.train_cnn --manifest data/fire_manifest.csv --epochs-head 5 --epochs-ft 5 --seed 42

## 5. Read the result and download the files

In [ ]:
import json
print(json.dumps(json.load(open("models/metrics/cnn_metrics.json")), indent=2))

### Export portable weights (so the model also loads on a different TensorFlow version)

In [ ]:
!python -m ml.export_fire_weights --model models/fire_cnn.keras

In [ ]:
from google.colab import files
for p in ["models/fire_cnn.keras", "models/fire_cnn_weights.npz", "models/fire_cnn_meta.json", "models/metrics/cnn_metrics.json"]:
    files.download(p)

## 6. After downloading
Copy the three files into the same folders of your local project, install the **same** TensorFlow version
(`pip install -r requirements-ml.txt`), and try it without a webcam:
`python -m src.main --sim --scenario fire --camera path\\to\\fire_video.mp4`.
Record the test F1, recall, confusion matrix and chosen threshold in Table 6.8 of the report.